# MARTA Bus Performance Deep Dive
## Headway Analysis & Pre/Post NextGen Network Comparison

**Author:** Aidan Moran  
**Date:** May 2026  
**Data Source:** [MARTA GTFS Static Feeds](https://itsmarta.com/app-developer-resources.aspx) via [Mobility Database](https://mobilitydatabase.org/feeds/gtfs/mdb-368)

---

On **April 18, 2026**, MARTA launched its **NextGen Bus Network** — the largest bus network redesign since the 1970s. 
This notebook analyzes bus service performance before and after the redesign, focusing on:

1. **Route Profiles** — length, stop count, stop spacing, and service levels for every bus route
2. **Headway Analysis** — scheduled headway distributions by route, including variability (std dev)
3. **Pre vs. Post NextGen Comparison** — how the redesign changed service frequency, coverage, and headway patterns
4. **Bus Bunching Risk** — identifying routes where scheduled headways are irregular
5. **Seasonal Variation** — how service levels shift across summer, fall, and winter

> **Future Enhancement:** A real-time (GTFS-RT) data collection pipeline to measure *actual* vs. scheduled headways and quantify bus bunching in practice.

## 1. Setup

Import libraries and configure the analysis environment.

In [ ]:
import os
import io
import zipfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from math import radians, cos, sin, asin, sqrt

# ── MARTA brand palette ──
MARTA_GOLD   = '#D4A843'
MARTA_RED    = '#CE202F'
MARTA_GREEN  = '#009B3A'
MARTA_BLUE   = '#0072CE'
MARTA_DARK   = '#1C1C1C'
MARTA_GRAY   = '#6C757D'
PALETTE = [MARTA_BLUE, MARTA_GOLD, MARTA_RED, MARTA_GREEN, '#5B2C6F', '#E67E22', '#1ABC9C', '#E74C3C']

sns.set_theme(style='whitegrid', palette=PALETTE)
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.family': 'sans-serif',
})

print('Setup complete.')

## 2. Data Acquisition

We download **two GTFS static feeds** from the [Mobility Database](https://mobilitydatabase.org/feeds/gtfs/mdb-368) archive:

| Period | Feed Date | Service Dates | Description |
|--------|-----------|---------------|-------------|
| **Pre-NextGen** | Jan 26, 2026 | Dec 27, 2025 – Apr 18, 2026 | Last version of the legacy bus network |
| **Post-NextGen** | Apr 19, 2026 | Apr 18 – Aug 22, 2026 | New NextGen bus network |

GTFS (General Transit Feed Specification) is the industry standard for publishing transit schedules. 
Each feed contains route definitions, individual trips, stop-level arrival times, stop locations, and geographic route shapes.

In [ ]:
# ── Feed URLs ──
FEEDS = {
    'pre':  'https://files.mobilitydatabase.org/mdb-368/mdb-368-202601260114/mdb-368-202601260114.zip',
    'post': 'https://files.mobilitydatabase.org/mdb-368/mdb-368-202604190110/mdb-368-202604190110.zip',
}

DATA_ROOT = os.path.join('..', 'data', 'raw')

def download_gtfs(label, url):
    """Download and extract a GTFS feed if not already present."""
    dest = os.path.join(DATA_ROOT, f'marta_gtfs_{label}')
    os.makedirs(dest, exist_ok=True)
    if os.path.exists(os.path.join(dest, 'routes.txt')):
        print(f'  [{label}] Already downloaded -> {dest}')
        return dest
    print(f'  [{label}] Downloading from Mobility Database...')
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        zf.extractall(dest)
    print(f'  [{label}] Extracted {len(os.listdir(dest))} files -> {dest}')
    return dest

paths = {}
for label, url in FEEDS.items():
    paths[label] = download_gtfs(label, url)

print('\nDone. Both feeds ready.')

## 3. Data Loading

Load the core GTFS tables for both feeds. We filter to **bus routes only** (GTFS `route_type == 3`) 
since this analysis focuses on bus performance and the NextGen redesign only affected buses.

In [ ]:
def load_gtfs(path, label):
    """Load core GTFS tables and filter to bus routes."""
    routes     = pd.read_csv(os.path.join(path, 'routes.txt'), dtype=str)
    trips      = pd.read_csv(os.path.join(path, 'trips.txt'), dtype=str)
    stop_times = pd.read_csv(os.path.join(path, 'stop_times.txt'), dtype=str)
    stops      = pd.read_csv(os.path.join(path, 'stops.txt'), dtype=str)
    calendar   = pd.read_csv(os.path.join(path, 'calendar.txt'), dtype=str)
    
    shapes_path = os.path.join(path, 'shapes.txt')
    shapes = pd.read_csv(shapes_path, dtype=str) if os.path.exists(shapes_path) else None
    
    bus_routes = routes[routes['route_type'] == '3'].copy()
    bus_route_ids = set(bus_routes['route_id'])
    bus_trips = trips[trips['route_id'].isin(bus_route_ids)].copy()
    bus_trip_ids = set(bus_trips['trip_id'])
    bus_stop_times = stop_times[stop_times['trip_id'].isin(bus_trip_ids)].copy()
    
    def time_to_minutes(t):
        try:
            parts = t.split(':')
            return int(parts[0]) * 60 + int(parts[1]) + int(parts[2]) / 60
        except:
            return np.nan
    
    bus_stop_times['arrival_min'] = bus_stop_times['arrival_time'].apply(time_to_minutes)
    bus_stop_times['departure_min'] = bus_stop_times['departure_time'].apply(time_to_minutes)
    bus_stop_times['stop_sequence'] = bus_stop_times['stop_sequence'].astype(int)
    stops['stop_lat'] = stops['stop_lat'].astype(float)
    stops['stop_lon'] = stops['stop_lon'].astype(float)
    
    print(f'[{label}] {len(bus_routes)} bus routes, {len(bus_trips)} trips, '
          f'{len(bus_stop_times):,} stop_times')
    
    return {
        'routes': bus_routes, 'trips': bus_trips, 'stop_times': bus_stop_times,
        'stops': stops, 'calendar': calendar, 'shapes': shapes, 'all_routes': routes,
    }

data = {}
for label, path in paths.items():
    data[label] = load_gtfs(path, label)

### Weekday Service

We focus on **weekday service** as it represents the primary commuter network and is the standard basis for transit performance evaluation.

In [ ]:
def get_weekday_trips(d):
    cal = d['calendar']
    weekday_services = cal[cal['monday'] == '1']['service_id'].values
    return d['trips'][d['trips']['service_id'].isin(weekday_services)].copy()

for label in ['pre', 'post']:
    wk_trips = get_weekday_trips(data[label])
    data[label]['weekday_trips'] = wk_trips
    wk_trip_ids = set(wk_trips['trip_id'])
    data[label]['weekday_stop_times'] = data[label]['stop_times'][
        data[label]['stop_times']['trip_id'].isin(wk_trip_ids)
    ].copy()
    print(f'[{label}] Weekday: {len(wk_trips)} trips, '
          f'{len(data[label]["weekday_stop_times"]):,} stop_times')

## 4. Route Profiles

For each bus route, we compute a comprehensive profile:

- **Route name** and short name (number)
- **Primary area** — the neighborhood/location of the route's midpoint
- **Route length** — total distance in miles (computed from stop coordinates)
- **Stop count** — number of unique stops served
- **Stops per mile** — stop density (a measure of local vs. express service)
- **Weekday trips** — total daily trips scheduled
- **Revenue hours** — total scheduled vehicle-hours per day (a better proxy for service investment than raw trip count, since it weights longer routes appropriately)

We compute this for both the pre- and post-NextGen networks.

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """Calculate distance in miles between two lat/lon points."""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 2 * 3956 * asin(sqrt(a))


def compute_route_length(route_id, trips, stop_times, stops):
    """Estimate route length from the longest trip's stop sequence."""
    route_trips = trips[trips['route_id'] == route_id]
    if route_trips.empty:
        return 0.0
    trip_stop_counts = stop_times[stop_times['trip_id'].isin(route_trips['trip_id'])]\
        .groupby('trip_id').size()
    if trip_stop_counts.empty:
        return 0.0
    longest_trip = trip_stop_counts.idxmax()
    trip_stops = stop_times[stop_times['trip_id'] == longest_trip]\
        .sort_values('stop_sequence')\
        .merge(stops[['stop_id', 'stop_lat', 'stop_lon']], on='stop_id')
    if len(trip_stops) < 2:
        return 0.0
    total_dist = 0.0
    for i in range(1, len(trip_stops)):
        total_dist += haversine(
            trip_stops.iloc[i-1]['stop_lat'], trip_stops.iloc[i-1]['stop_lon'],
            trip_stops.iloc[i]['stop_lat'], trip_stops.iloc[i]['stop_lon']
        )
    return total_dist


def build_route_profiles(d, label):
    """Build a profile DataFrame for all bus routes."""
    routes = d['routes']
    trips = d['weekday_trips']
    st = d['weekday_stop_times']
    stops = d['stops']
    
    profiles = []
    for _, route in routes.iterrows():
        rid = route['route_id']
        rname = route.get('route_long_name', route.get('route_short_name', rid))
        rshort = route.get('route_short_name', rid)
        
        route_trips = trips[trips['route_id'] == rid]
        n_trips = len(route_trips)
        
        route_st = st[st['trip_id'].isin(route_trips['trip_id'])]
        unique_stops = route_st['stop_id'].nunique()
        
        # Revenue hours: sum of (last arrival - first departure) for each trip
        trip_durations = route_st.groupby('trip_id').agg(
            trip_start=('departure_min', 'min'),
            trip_end=('arrival_min', 'max')
        )
        trip_durations['duration_hrs'] = (trip_durations['trip_end'] - trip_durations['trip_start']) / 60
        revenue_hrs = trip_durations['duration_hrs'].sum()
        
        length = compute_route_length(rid, trips, st, stops)
        stops_per_mile = unique_stops / length if length > 0.5 else np.nan
        
        # Primary area — find the stop closest to the route centroid
        stop_ids_on_route = route_st['stop_id'].unique()
        route_stops = stops[stops['stop_id'].isin(stop_ids_on_route)]
        if not route_stops.empty:
            mid_lat = route_stops['stop_lat'].mean()
            mid_lon = route_stops['stop_lon'].mean()
            dists = route_stops.apply(
                lambda s: haversine(mid_lat, mid_lon, s['stop_lat'], s['stop_lon']), axis=1
            )
            center_stop = route_stops.loc[dists.idxmin()]
            area = center_stop['stop_name']
        else:
            area = 'Unknown'
            mid_lat = mid_lon = np.nan
        
        profiles.append({
            'route_id': rid,
            'route_short_name': rshort,
            'route_name': rname,
            'area': area,
            'length_mi': round(length, 2),
            'n_stops': unique_stops,
            'stops_per_mi': round(stops_per_mile, 1) if not np.isnan(stops_per_mile) else np.nan,
            'weekday_trips': n_trips,
            'revenue_hrs': round(revenue_hrs, 1),
            'center_lat': mid_lat,
            'center_lon': mid_lon,
            'period': label,
        })
    
    return pd.DataFrame(profiles)

print('Building route profiles (this may take a minute)...')
profiles = {}
for label in ['pre', 'post']:
    profiles[label] = build_route_profiles(data[label], label)
    print(f'  [{label}] {len(profiles[label])} bus routes profiled')

print('Done.')

In [ ]:
# Display the post-NextGen route profiles, sorted by revenue hours
cols = ['route_short_name', 'route_name', 'area', 'length_mi', 'n_stops', 'stops_per_mi', 'weekday_trips', 'revenue_hrs']
print(f"{'='*100}")
print(f"POST-NEXTGEN BUS ROUTE PROFILES (Top 20 by Revenue Hours)")
print(f"{'='*100}")
profiles['post'].sort_values('revenue_hrs', ascending=False)[cols].head(20)\
    .style.background_gradient(subset=['revenue_hrs'], cmap='YlOrRd')\
    .format({'length_mi': '{:.1f}', 'stops_per_mi': '{:.1f}', 'revenue_hrs': '{:.1f}'})

## 5. Headway Analysis

**Headway** is the time between consecutive buses on the same route at the same stop. 
It's the single most important metric riders experience — a 10-minute headway means a bus every 10 minutes.

We calculate headways by:
1. For each route + direction, finding the **first stop** in the sequence
2. Sorting departures chronologically at that stop
3. Computing the time gap between consecutive departures

A route with **low mean headway** and **low standard deviation** provides reliable, frequent service. 
High standard deviation signals irregular spacing — a precursor to bus bunching.

In [ ]:
def compute_headways(d, label):
    """Compute headways for all bus routes."""
    trips = d['weekday_trips']
    st = d['weekday_stop_times']
    routes = d['routes']
    st_merged = st.merge(trips[['trip_id', 'route_id', 'direction_id']], on='trip_id')
    
    all_headways = []
    for rid in routes['route_id'].unique():
        route_st = st_merged[st_merged['route_id'] == rid]
        for direction in route_st['direction_id'].dropna().unique():
            dir_st = route_st[route_st['direction_id'] == direction]
            first_stops = dir_st.loc[dir_st.groupby('trip_id')['stop_sequence'].idxmin()]
            if first_stops.empty:
                continue
            common_first = first_stops['stop_id'].mode()
            if common_first.empty:
                continue
            departures = first_stops[first_stops['stop_id'] == common_first.iloc[0]]\
                .sort_values('departure_min')['departure_min'].values
            if len(departures) < 2:
                continue
            headways = np.diff(departures)
            headways = headways[(headways > 0) & (headways <= 180)]
            for hw in headways:
                all_headways.append({'route_id': rid, 'direction_id': direction,
                                     'headway_min': hw, 'period': label})
    return pd.DataFrame(all_headways)

print('Computing headways...')
headways = {}
for label in ['pre', 'post']:
    headways[label] = compute_headways(data[label], label)
    print(f'  [{label}] {len(headways[label]):,} headway observations across '
          f'{headways[label]["route_id"].nunique()} routes')
print('Done.')

In [ ]:
def headway_stats(hw_df, profile_df):
    """Compute summary headway statistics per route."""
    stats = hw_df.groupby('route_id')['headway_min'].agg(
        headway_mean='mean', headway_median='median', headway_std='std',
        headway_min='min', headway_max='max', headway_count='count',
    ).round(1).reset_index()
    stats['headway_cv'] = (stats['headway_std'] / stats['headway_mean']).round(2)
    return profile_df.merge(stats, on='route_id', how='left')

route_stats = {}
for label in ['pre', 'post']:
    route_stats[label] = headway_stats(headways[label], profiles[label])
    print(f'[{label}] Route stats computed for {len(route_stats[label])} routes')

display_cols = ['route_short_name', 'route_name', 'weekday_trips', 'revenue_hrs',
                'headway_mean', 'headway_median', 'headway_std', 'headway_cv',
                'headway_min', 'headway_max']
print(f"\n{'='*110}")
print('POST-NEXTGEN HEADWAY STATISTICS (Top 15 by Revenue Hours)')
print(f"{'='*110}")
route_stats['post'].sort_values('revenue_hrs', ascending=False)[display_cols].head(15)

## 6. Headway Distributions — Top Routes

Violin plots show the full distribution of scheduled headways for the busiest routes. 
A tight, narrow violin means consistent, reliable service. 
A wide or multi-peaked violin reveals irregular spacing — the kind that leads to bus bunching.

In [ ]:
def plot_headway_violins(hw_df, stats_df, label, n=10):
    """Violin plot of headway distributions for top N routes by revenue hours."""
    top_routes = stats_df.sort_values('revenue_hrs', ascending=False).head(n)
    top_ids = top_routes['route_id'].values
    plot_data = hw_df[hw_df['route_id'].isin(top_ids)].merge(
        top_routes[['route_id', 'route_short_name', 'route_name', 'revenue_hrs']],
        on='route_id'
    )
    plot_data['label'] = plot_data['route_short_name'] + ' \u2014 ' + plot_data['route_name']
    order = plot_data.groupby('label')['revenue_hrs'].first().sort_values(ascending=True).index
    
    fig, ax = plt.subplots(figsize=(12, 8))
    parts = ax.violinplot(
        [plot_data[plot_data['label'] == lbl]['headway_min'].values for lbl in order],
        positions=range(len(order)), vert=False, showmeans=True, showmedians=True,
    )
    for i, body in enumerate(parts['bodies']):
        body.set_facecolor(PALETTE[i % len(PALETTE)])
        body.set_alpha(0.7)
    parts['cmeans'].set_color(MARTA_DARK)
    parts['cmedians'].set_color(MARTA_RED)
    for k in ['cmins', 'cmaxes', 'cbars']:
        parts[k].set_color(MARTA_GRAY)
    
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(order, fontsize=10)
    ax.set_xlabel('Headway (minutes)', fontsize=12)
    period_label = 'Post-NextGen' if label == 'post' else 'Pre-NextGen'
    ax.set_title(f'Scheduled Headway Distribution \u2014 Top {n} Routes by Revenue Hours ({period_label})',
                 fontsize=14, fontweight='bold')
    ax.axvline(x=15, color=MARTA_GREEN, linestyle='--', alpha=0.5, label='15-min frequency target')
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(os.path.join('..', 'assets', f'headway_violins_{label}.png'), dpi=150, bbox_inches='tight')
    plt.show()

os.makedirs(os.path.join('..', 'assets'), exist_ok=True)
plot_headway_violins(headways['post'], route_stats['post'], 'post', n=10)

In [ ]:
# Pre-NextGen violin plots for comparison
plot_headway_violins(headways['pre'], route_stats['pre'], 'pre', n=10)

## 7. Pre vs. Post NextGen Comparison

The NextGen Bus Network was the most comprehensive MARTA bus redesign since the 1970s. Let's quantify what changed.

In [ ]:
# ── Network-level summary ──
print(f"{'='*70}")
print(f"{'METRIC':<35} {'PRE':>10} {'POST':>10} {'CHANGE':>10}")
print(f"{'='*70}")

metrics = [
    ('Bus routes', len(profiles['pre']), len(profiles['post'])),
    ('Weekday trips (all bus)', len(data['pre']['weekday_trips']), len(data['post']['weekday_trips'])),
    ('Total revenue hours', profiles['pre']['revenue_hrs'].sum(), profiles['post']['revenue_hrs'].sum()),
    ('Unique stops served', data['pre']['weekday_stop_times']['stop_id'].nunique(),
     data['post']['weekday_stop_times']['stop_id'].nunique()),
    ('Avg headway (min)', headways['pre']['headway_min'].mean(), headways['post']['headway_min'].mean()),
    ('Median headway (min)', headways['pre']['headway_min'].median(), headways['post']['headway_min'].median()),
    ('Headway std dev (min)', headways['pre']['headway_min'].std(), headways['post']['headway_min'].std()),
]

for name, pre, post in metrics:
    if isinstance(pre, float):
        print(f"{name:<35} {pre:>10.1f} {post:>10.1f} {post - pre:>+10.1f}")
    else:
        print(f"{name:<35} {pre:>10,} {post:>10,} {post - pre:>+10,}")
print(f"{'='*70}")

In [ ]:
# ── Side-by-side headway distribution ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, label, color, title in zip(axes, ['pre', 'post'],
                                     [MARTA_GRAY, MARTA_BLUE],
                                     ['Pre-NextGen', 'Post-NextGen']):
    hw = headways[label]['headway_min']
    ax.hist(hw, bins=np.arange(0, 125, 5), color=color, alpha=0.75, edgecolor='white')
    ax.axvline(hw.mean(), color=MARTA_RED, linestyle='--', linewidth=2, label=f'Mean: {hw.mean():.1f} min')
    ax.axvline(hw.median(), color=MARTA_GOLD, linestyle='-', linewidth=2, label=f'Median: {hw.median():.1f} min')
    ax.set_xlabel('Headway (minutes)')
    ax.set_title(title, fontweight='bold')
    ax.legend()
axes[0].set_ylabel('Count')
fig.suptitle('Headway Distribution: Before & After NextGen Redesign', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join('..', 'assets', 'headway_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Route changes: added, removed, retained ──
pre_routes = set(profiles['pre']['route_short_name'])
post_routes = set(profiles['post']['route_short_name'])
added = post_routes - pre_routes
removed = pre_routes - post_routes
retained = pre_routes & post_routes
print(f'Routes ADDED in NextGen:   {len(added):>3}  \u2014 {", ".join(sorted(added)[:15])}{"..." if len(added) > 15 else ""}')
print(f'Routes REMOVED:            {len(removed):>3}  \u2014 {", ".join(sorted(removed)[:15])}{"..." if len(removed) > 15 else ""}')
print(f'Routes RETAINED:           {len(retained):>3}')
print(f'\nNet change: {len(post_routes)} - {len(pre_routes)} = {len(post_routes) - len(pre_routes):+d} routes')

### Headway Changes on Retained Routes

For routes that exist in both the pre- and post-NextGen networks, how did headways change?

In [ ]:
pre_s = route_stats['pre'][route_stats['pre']['route_short_name'].isin(retained)]\
    [['route_short_name', 'route_name', 'weekday_trips', 'revenue_hrs', 'headway_mean', 'headway_std']]\
    .rename(columns={'weekday_trips': 'trips_pre', 'revenue_hrs': 'revhrs_pre',
                     'headway_mean': 'hw_mean_pre', 'headway_std': 'hw_std_pre'})

post_s = route_stats['post'][route_stats['post']['route_short_name'].isin(retained)]\
    [['route_short_name', 'weekday_trips', 'revenue_hrs', 'headway_mean', 'headway_std']]\
    .rename(columns={'weekday_trips': 'trips_post', 'revenue_hrs': 'revhrs_post',
                     'headway_mean': 'hw_mean_post', 'headway_std': 'hw_std_post'})

comparison = pre_s.merge(post_s, on='route_short_name', how='inner')
comparison['hw_change'] = (comparison['hw_mean_post'] - comparison['hw_mean_pre']).round(1)
comparison['trips_change'] = comparison['trips_post'] - comparison['trips_pre']
comparison['revhrs_change'] = (comparison['revhrs_post'] - comparison['revhrs_pre']).round(1)

print('Routes with BIGGEST HEADWAY IMPROVEMENTS (more frequent service):')
print('='*90)
display(comparison.sort_values('hw_change').head(10)[
    ['route_short_name', 'route_name', 'trips_pre', 'trips_post',
     'revhrs_pre', 'revhrs_post', 'hw_mean_pre', 'hw_mean_post', 'hw_change']
])

print('\nRoutes with BIGGEST HEADWAY INCREASES (less frequent service):')
print('='*90)
display(comparison.sort_values('hw_change', ascending=False).head(10)[
    ['route_short_name', 'route_name', 'trips_pre', 'trips_post',
     'revhrs_pre', 'revhrs_post', 'hw_mean_pre', 'hw_mean_post', 'hw_change']
])

In [ ]:
# ── Scatter: headway change vs. revenue hour change ──
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(comparison['hw_change'], comparison['revhrs_change'],
           s=80, alpha=0.7, c=MARTA_BLUE, edgecolors='white', linewidth=0.5)

for _, row in comparison.nlargest(5, 'revhrs_change').iterrows():
    ax.annotate(row['route_short_name'], (row['hw_change'], row['revhrs_change']),
                fontsize=9, fontweight='bold', xytext=(5, 5), textcoords='offset points')
for _, row in comparison.nsmallest(5, 'revhrs_change').iterrows():
    ax.annotate(row['route_short_name'], (row['hw_change'], row['revhrs_change']),
                fontsize=9, fontweight='bold', xytext=(5, -5), textcoords='offset points')

ax.axhline(0, color=MARTA_GRAY, linestyle='-', alpha=0.3)
ax.axvline(0, color=MARTA_GRAY, linestyle='-', alpha=0.3)
ax.text(0.02, 0.98, 'More rev. hours, shorter headways\n(improved)', transform=ax.transAxes,
        va='top', ha='left', fontsize=9, color=MARTA_GREEN, style='italic')
ax.text(0.98, 0.02, 'Fewer rev. hours, longer headways\n(reduced)', transform=ax.transAxes,
        va='bottom', ha='right', fontsize=9, color=MARTA_RED, style='italic')
ax.set_xlabel('Headway Change (minutes) \u2014 negative = more frequent', fontsize=12)
ax.set_ylabel('Revenue Hours Change (per day)', fontsize=12)
ax.set_title('NextGen Impact on Retained Bus Routes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join('..', 'assets', 'nextgen_route_changes.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Bus Bunching Risk Assessment

**Bus bunching** occurs when buses on the same route cluster together, creating long gaps followed by multiple arrivals in quick succession. 
Even in scheduled data, we can identify routes *prone* to bunching by looking at:

- **Coefficient of Variation (CV)** = std / mean — higher CV means more irregular headways
- **Max/Min ratio** — how extreme the headway swings are

Routes with high CV in the *schedule* are almost guaranteed to experience bunching in practice, 
since real-world delays only compound the irregularity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, label, title in zip(axes, ['pre', 'post'], ['Pre-NextGen', 'Post-NextGen']):
    rs = route_stats[label].dropna(subset=['headway_cv', 'headway_mean'])
    rs = rs[rs['headway_count'] >= 5]
    ax.scatter(rs['headway_mean'], rs['headway_cv'], s=rs['revenue_hrs'] * 0.8,
              alpha=0.6, c=MARTA_BLUE if label == 'post' else MARTA_GRAY,
              edgecolors='white', linewidth=0.5)
    high_risk = rs[rs['headway_cv'] > 0.5]
    for _, row in high_risk.nlargest(5, 'revenue_hrs').iterrows():
        ax.annotate(row['route_short_name'], (row['headway_mean'], row['headway_cv']),
                    fontsize=8, fontweight='bold', color=MARTA_RED)
    ax.axhline(0.5, color=MARTA_RED, linestyle='--', alpha=0.4, label='High variability threshold')
    ax.set_xlabel('Mean Headway (minutes)')
    ax.set_ylabel('Coefficient of Variation')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9)
fig.suptitle('Headway Regularity by Route (bubble size = revenue hours)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join('..', 'assets', 'bunching_risk.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
risk = route_stats['post'].dropna(subset=['headway_cv'])
risk = risk[risk['headway_count'] >= 5].sort_values('headway_cv', ascending=False)
print('POST-NEXTGEN: Routes with Highest Scheduled Headway Variability')
print('(Most prone to bus bunching in real-world operations)')
print('='*100)
risk[['route_short_name', 'route_name', 'weekday_trips', 'revenue_hrs',
      'headway_mean', 'headway_std', 'headway_cv', 'headway_min', 'headway_max']].head(15)

## 9. Seasonal Service Variation

Does MARTA adjust bus schedules seasonally? Using archived GTFS feeds from the Mobility Database, we compare 
scheduled service across seasons on the **pre-NextGen network**.

| Season | Feed Date | Service Dates |
|--------|-----------|---------------|
| **Summer 2025** | Jun 17, 2025 | Jun 16 – Aug 23, 2025 |
| **Fall 2025** | Oct 4, 2025 | Oct 4 – Dec 27, 2025 |
| **Winter 2025-26** | Dec 27, 2025 | Dec 27, 2025 – Apr 18, 2026 |
| **Spring 2026** | Jan 26, 2026 | Dec 27, 2025 – Apr 18, 2026 |

In [ ]:
SEASONAL_FEEDS = {
    'summer_2025': 'https://files.mobilitydatabase.org/mdb-368/mdb-368-202506170118/mdb-368-202506170118.zip',
    'fall_2025':   'https://files.mobilitydatabase.org/mdb-368/mdb-368-202510040001/mdb-368-202510040001.zip',
    'winter_2025': 'https://files.mobilitydatabase.org/mdb-368/mdb-368-202512270033/mdb-368-202512270033.zip',
}
seasonal_paths = {}
for label, url in SEASONAL_FEEDS.items():
    seasonal_paths[label] = download_gtfs(label, url)
print('\nAll seasonal feeds downloaded.')

In [ ]:
seasonal_data = {}
seasonal_headways = {}
for label, path in seasonal_paths.items():
    d = load_gtfs(path, label)
    wk = get_weekday_trips(d)
    d['weekday_trips'] = wk
    d['weekday_stop_times'] = d['stop_times'][d['stop_times']['trip_id'].isin(set(wk['trip_id']))].copy()
    seasonal_data[label] = d
    seasonal_headways[label] = compute_headways(d, label)
    print(f'  [{label}] {len(wk)} weekday trips, {len(seasonal_headways[label]):,} headway obs')

seasonal_headways['spring_2026'] = headways['pre'].copy()
seasonal_data['spring_2026'] = data['pre']
print('\nSeasonal headways computed.')

In [ ]:
season_labels = {'summer_2025': 'Summer 2025', 'fall_2025': 'Fall 2025',
                  'winter_2025': 'Winter 2025-26', 'spring_2026': 'Spring 2026'}

print(f"{'='*75}")
print(f"{'SEASONAL SERVICE COMPARISON (Pre-NextGen Network)':^75}")
print(f"{'='*75}")
print(f"{'Metric':<25} {'Summer':>12} {'Fall':>12} {'Winter':>12} {'Spring':>12}")
print(f"{'-'*75}")
for name, fn in [
    ('Weekday trips', lambda l: len(seasonal_data[l]['weekday_trips'])),
    ('Bus routes', lambda l: len(seasonal_data[l]['routes'])),
    ('Avg headway (min)', lambda l: seasonal_headways[l]['headway_min'].mean()),
    ('Median headway (min)', lambda l: seasonal_headways[l]['headway_min'].median()),
    ('Headway std dev', lambda l: seasonal_headways[l]['headway_min'].std()),
]:
    vals = [fn(k) for k in ['summer_2025', 'fall_2025', 'winter_2025', 'spring_2026']]
    if isinstance(vals[0], float):
        print(f"{name:<25} {vals[0]:>12.1f} {vals[1]:>12.1f} {vals[2]:>12.1f} {vals[3]:>12.1f}")
    else:
        print(f"{name:<25} {vals[0]:>12,} {vals[1]:>12,} {vals[2]:>12,} {vals[3]:>12,}")
print(f"{'='*75}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
season_colors = {'summer_2025': '#E67E22', 'fall_2025': '#A0522D',
                  'winter_2025': MARTA_BLUE, 'spring_2026': MARTA_GREEN}
bins = np.arange(0, 125, 5)
for key, label in season_labels.items():
    hw = seasonal_headways[key]['headway_min']
    ax.hist(hw, bins=bins, alpha=0.35, label=f'{label} (mean: {hw.mean():.1f} min)',
            color=season_colors[key], edgecolor='white', linewidth=0.5)
ax.set_xlabel('Headway (minutes)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Seasonal Headway Distributions (Pre-NextGen Network)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join('..', 'assets', 'seasonal_headways.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Each GTFS feed version uses different route_id numbers (e.g., summer Route 1 = 25949,
# fall = 26773, winter = 27323), so we must join on route_short_name instead of route_id.

seasonal_route_stats = {}
for key in ['summer_2025', 'fall_2025', 'winter_2025', 'spring_2026']:
    # Map route_id -> route_short_name for this feed
    if key == 'spring_2026':
        route_map = data['pre']['routes'][['route_id', 'route_short_name']]
    else:
        route_map = seasonal_data[key]['routes'][['route_id', 'route_short_name']]

    stats = seasonal_headways[key].groupby('route_id')['headway_min'].agg(
        headway_mean='mean', headway_std='std', trip_count='count'
    ).round(1).reset_index()

    # Join route_short_name and use it as the index
    stats = stats.merge(route_map, on='route_id', how='left')
    stats = stats.set_index('route_short_name').drop(columns=['route_id'])
    stats.columns = [f'{c}_{key}' for c in stats.columns]
    seasonal_route_stats[key] = stats

seasonal_merged = seasonal_route_stats['summer_2025']
for key in ['fall_2025', 'winter_2025', 'spring_2026']:
    seasonal_merged = seasonal_merged.join(seasonal_route_stats[key], how='outer')

# Add route long names from any feed (pre-NextGen)
route_names = data['pre']['routes'][['route_short_name', 'route_long_name']].drop_duplicates()
seasonal_merged = seasonal_merged.reset_index().merge(route_names, on='route_short_name', how='left')
hw_cols = [c for c in seasonal_merged.columns if c.startswith('headway_mean_')]
seasonal_merged['seasonal_range'] = seasonal_merged[hw_cols].max(axis=1) - seasonal_merged[hw_cols].min(axis=1)

print('Routes with LARGEST Seasonal Headway Variation:')
print('='*90)
display(seasonal_merged.sort_values('seasonal_range', ascending=False).head(15)[
    ['route_short_name', 'route_long_name',
     'headway_mean_summer_2025', 'headway_mean_fall_2025',
     'headway_mean_winter_2025', 'headway_mean_spring_2026',
     'seasonal_range']
].rename(columns={
    'headway_mean_summer_2025': 'Summer', 'headway_mean_fall_2025': 'Fall',
    'headway_mean_winter_2025': 'Winter', 'headway_mean_spring_2026': 'Spring',
    'seasonal_range': 'Range (min)'
}))

### Seasonality Takeaways

The seasonal comparison reveals whether MARTA adjusts service levels throughout the year:

- **Do headways widen in summer?** Some agencies reduce frequency when school is out and ridership dips.
- **Which routes see the biggest seasonal swings?** Routes serving schools/universities may change dramatically.
- **Is winter service different from fall?** Holiday schedules and weather could affect planned service.

## 10. Complete Route Reference

Full route profile table with all metrics, exportable as a reference.

In [ ]:
output_dir = os.path.join('..', 'data', 'processed')
os.makedirs(output_dir, exist_ok=True)
for label in ['pre', 'post']:
    out_path = os.path.join(output_dir, f'marta_bus_route_stats_{label}_nextgen.csv')
    route_stats[label].to_csv(out_path, index=False)
    print(f'Saved: {out_path}')
comp_path = os.path.join(output_dir, 'marta_nextgen_route_comparison.csv')
comparison.to_csv(comp_path, index=False)
print(f'Saved: {comp_path}')

## 11. Future Work: Real-Time Bus Tracking (GTFS-RT)

This analysis uses **scheduled** GTFS data, which reveals service design intent. 
To measure *actual* bus bunching — like the "zero buses for 30 minutes, then three at once" phenomenon — 
we need **GTFS-Realtime** data.

### Data Collection Plan

MARTA publishes a GTFS-RT Vehicle Positions feed at:
```
https://gtfs-rt.itsmarta.com/TMGTFSRealTimeWebService/vehicle/vehiclepositions.pb
```

**Approach:**
1. Set up a scheduled job to poll vehicle positions every 30 seconds
2. Store timestamped position records for 2-4 weeks
3. Reconstruct actual arrival times at stops from position interpolation
4. Compare actual headways to scheduled headways
5. Quantify bunching: percentage of headways < 2 min (bunched) vs. > 2x scheduled (gapped)

### Expected Findings

Routes with high scheduled CV (identified above) are the strongest candidates for real-world bunching. 
The RT data would let us answer:
- How often does bunching occur on the worst routes?
- What time of day is bunching worst?
- Has NextGen improved or worsened actual headway reliability?

## 12. Key Findings

### Network Transformation
The NextGen redesign represented a fundamental restructuring of MARTA's bus network. This analysis compared 
the pre-NextGen schedule (effective through April 17, 2026) with the post-NextGen schedule (effective April 18, 2026) 
across route structure, service levels, and headway regularity.

### Methodology Notes
- **Headways** are computed from scheduled departure times at each route's first stop, per direction
- **Route length** is estimated from the geographic distance between sequential stops on the longest trip
- **Revenue hours** (total scheduled vehicle-hours per route per day) serve as a service investment proxy; 
this aligns with MARTA's own Bus Productivity Index methodology, which uses passengers per in-service hour 
as a core metric (FY2026 Service Standards Report, Section 4.5)
- **Coefficient of Variation** (CV = sigma/mu) measures headway regularity independent of frequency level

### Limitations
- Scheduled headways reflect service *design*, not operational reality — actual performance requires GTFS-RT data
- The post-NextGen network is only 3 weeks old; schedule adjustments are likely as MARTA refines the new network
- Route-level ridership data is not publicly available; revenue hours are an imperfect proxy for demand

---
*Analysis by Aidan Moran | Data from MARTA via Mobility Database | May 2026*